# Model Audit — Reproduction (v0.1 scorecard)
Reproduces every figure cited in AUDIT.md for the audited v0.1 loan-default scorecard.

Run from the repo root:  python audit/reproduce_audit.py
Requires pandas, numpy, scikit-learn. Re-runs the modelling cells of the audited notebook
(audit/Loan_Default_Score_Engine_v0.1.ipynb, kept unchanged as audit evidence), then runs the
audit probes. The CSV is read from loan-default-scorecard/data/ if present, else from the zip.

In [1]:
import contextlib, io, json, os, warnings, zipfile
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
CSV = 'loan-default-scorecard/data/Loan_Default.csv'
RAW = pd.read_csv(CSV if os.path.exists(CSV) else zipfile.ZipFile('loan-default-scorecard.zip').open(CSV))


def section(t):
    print(f'\n=== {t} ===')

## 1. label vs missingness

In [2]:
section('1. Default rate by missingness (raw data)')
for c in RAW.columns[RAW.isna().any()]:
    m = RAW[c].isna()
    print(f'{c:28s} missing={m.mean():6.3f}  P(def|missing)={RAW.Status[m].mean():.3f}  '
          f'P(def|present)={RAW.Status[~m].mean():.3f}')
rate_m, pv_m = RAW.rate_of_interest.isna(), RAW.property_value.isna()
equi = RAW.credit_type == 'EQUI'
flag = rate_m | pv_m | equi
print('LTV missing == property_value missing:', bool((RAW.LTV.isna() == pv_m).all()))
print(f'P(def | none of rate-missing/PV-missing/EQUI) = {RAW.Status[~flag].mean():.4f}  (n={int((~flag).sum())})')
print(f'Defaults with rate_of_interest present: {int((RAW.Status.eq(1) & ~rate_m).sum())} of {int(RAW.Status.sum())}')
print('EQUI rows with property_value missing:', int((equi & pv_m).sum()), 'of', int(equi.sum()))


=== 1. Default rate by missingness (raw data) ===
loan_limit                   missing= 0.022  P(def|missing)=0.263  P(def|present)=0.246
approv_in_adv                missing= 0.006  P(def|missing)=0.265  P(def|present)=0.246
loan_purpose                 missing= 0.001  P(def|missing)=0.261  P(def|present)=0.246
rate_of_interest             missing= 0.245  P(def|missing)=1.000  P(def|present)=0.002
Interest_rate_spread         missing= 0.246  P(def|missing)=1.000  P(def|present)=0.000
Upfront_charges              missing= 0.267  P(def|missing)=0.920  P(def|present)=0.001
term                         missing= 0.000  P(def|missing)=0.366  P(def|present)=0.246
Neg_ammortization            missing= 0.001  P(def|missing)=0.264  P(def|present)=0.246
property_value               missing= 0.102  P(def|missing)=1.000  P(def|present)=0.161
income                       missing= 0.062  P(def|missing)=0.135  P(def|present)=0.254
age                          missing= 0.001  P(def|missing)=1.000  P(

## 2. rerun the notebook

In [3]:
section('2. Notebook re-run (cells unchanged, plots skipped)')
nb = json.load(open('audit/Loan_Default_Score_Engine_v0.1.ipynb'))
code = '\n'.join(''.join(c['source']) for c in nb['cells']
                 if c['cell_type'] == 'code' and 'plt.subplots' not in ''.join(c['source']))
code = code.replace("pd.read_csv(DATA_PATH)", "RAW.copy()")
ns = {'RAW': RAW}
with contextlib.redirect_stdout(io.StringIO()):
    exec(code, ns)
g = ns
p, y_test, X_test, X_train, y_train = g['p'], g['y_test'], g['X_test'], g['X_train'], g['y_train']
Xw_tr, Xw_test, cols, woe = g['Xw_tr'], g['Xw_test'], g['cols'], g['woe']
print(f'test AUC={roc_auc_score(y_test, p):.4f}  (notebook/report: 0.822)')
print('coefficients (WoE convention: every sign should be negative):')
for k, v in g['coef'].items():
    print(f'  {k:28s} {v:+.3f}{"   <-- sign reversed" if v > 0 else ""}')
print(f"LTV train mean-imputation value lands in bin (58.97, 72.49]: WoE = {woe['LTV'][[k for k in woe['LTV'] if str(k).startswith('(58')][0]]:.2f}")
print(f"property_value MISSING WoE = {woe['property_value']['MISSING']:.2f}")
print(f'share of test scored PD > 0.99: {(p > 0.99).mean():.3f}')
print('WoE-feature correlation business_or_commercial ~ loan_type:',
      round(Xw_tr['business_or_commercial'].corr(Xw_tr['loan_type']), 3))


=== 2. Notebook re-run (cells unchanged, plots skipped) ===


test AUC=0.8219  (notebook/report: 0.822)
coefficients (WoE convention: every sign should be negative):
  loan_amount                  +1.756   <-- sign reversed
  property_value               -0.755
  income                       -0.921
  LTV                          +0.084   <-- sign reversed
  dtir1                        -0.028
  Gender                       -0.357
  loan_type                    -0.415
  business_or_commercial       -0.515
  Neg_ammortization            -1.131
  lump_sum_payment             -1.044
  co-applicant_credit_type     +0.223   <-- sign reversed
  submission_of_application    -0.937
LTV train mean-imputation value lands in bin (58.97, 72.49]: WoE = -0.84
property_value MISSING WoE = -10.90
share of test scored PD > 0.99: 0.101
WoE-feature correlation business_or_commercial ~ loan_type: 0.979


## 3. leakage-adjusted variants

In [4]:
section('3. Leakage-adjusted AUC variants (test set)')


def fit_auc(c, Xtr, Xte, ytr, yte):
    m = LogisticRegression(max_iter=1000).fit(Xtr[c], ytr)
    return roc_auc_score(yte, m.predict_proba(Xte[c])[:, 1])


Xtr2, Xte2 = Xw_tr.copy(), Xw_test.copy()
Xtr2.loc[X_train.property_value.isna(), 'property_value'] = 0
Xte2.loc[X_test.property_value.isna(), 'property_value'] = 0
trm, tem = X_train.property_value.notna(), X_test.property_value.notna()
variants = {
    'a) PV MISSING bin neutralised (WoE=0), refit': fit_auc(cols, Xtr2, Xte2, y_train, y_test),
    'b) property_value dropped': fit_auc([c for c in cols if c != 'property_value'], Xw_tr, Xw_test, y_train, y_test),
    'c) property_value, LTV, dtir1 dropped': fit_auc([c for c in cols if c not in ('property_value', 'LTV', 'dtir1')],
                                                     Xw_tr, Xw_test, y_train, y_test),
    'd) PV-present rows only (train and test)': fit_auc(cols, Xw_tr[trm], Xw_test[tem], y_train[trm], y_test[tem]),
    'e) shipped model scored on PV-present test rows': roc_auc_score(y_test[tem], p[tem.values]),
}
for k, v in variants.items():
    print(f'{k:50s} AUC={v:.4f}')

ids = g['df'].loc[X_test.index, 'ID']
rate_missing_te = RAW.set_index('ID').loc[ids, 'rate_of_interest'].isna().values
print('PV-present test rows: corr(Status, rate_of_interest missing) =',
      round(np.corrcoef(y_test[tem], rate_missing_te[tem.values])[0, 1], 4))


=== 3. Leakage-adjusted AUC variants (test set) ===


a) PV MISSING bin neutralised (WoE=0), refit       AUC=0.7919
b) property_value dropped                          AUC=0.7691
c) property_value, LTV, dtir1 dropped              AUC=0.7264
d) PV-present rows only (train and test)           AUC=0.6969
e) shipped model scored on PV-present test rows    AUC=0.6969
PV-present test rows: corr(Status, rate_of_interest missing) = 1.0


## 4. calibration

In [5]:
section('4. Calibration by decile (test)')
cal = pd.DataFrame({'p': p, 'y': y_test.values})
cal['d'] = pd.qcut(cal.p.rank(method='first'), 10, labels=False)
print(cal.groupby('d').agg(pred=('p', 'mean'), obs=('y', 'mean')).round(3).T.to_string())


=== 4. Calibration by decile (test) ===
d         0      1      2      3      4      5      6      7      8      9
pred  0.058  0.073  0.091  0.112  0.129  0.149  0.184  0.234  0.415  0.999
obs   0.055  0.077  0.096  0.105  0.133  0.146  0.189  0.243  0.405  1.000


## 5. naive model

In [6]:
section('5. Naive model: every column, WoE with MISSING bins')
Xn = RAW.drop(columns=['Status', 'ID', 'year'])
a, b, ya, yb = train_test_split(Xn, RAW.Status, test_size=.2, stratify=RAW.Status, random_state=42)
A, B = pd.DataFrame(index=a.index), pd.DataFrame(index=b.index)
for c in a.columns:
    if pd.api.types.is_numeric_dtype(a[c]) and a[c].nunique() > 20:
        q = np.unique(np.quantile(a[c].dropna(), np.linspace(0, 1, 6))); q[0], q[-1] = -np.inf, np.inf
        s1, s2 = g['num_bin'](a[c], q), g['num_bin'](b[c], q)
    else:
        s1, s2 = a[c].astype('object').fillna('MISSING'), b[c].astype('object').fillna('MISSING')
    w = g['woe_table'](s1, ya)
    A[c], B[c] = s1.map(w).astype(float), s2.map(w).astype(float)
m = LogisticRegression(max_iter=2000).fit(A.fillna(0), ya)
print(f'naive test AUC = {roc_auc_score(yb, m.predict_proba(B.fillna(0))[:, 1]):.4f}')


=== 5. Naive model: every column, WoE with MISSING bins ===


naive test AUC = 1.0000


## 6. row drops

In [7]:
section('6. Rows removed by dropna filters')
drop_cols = ['term', 'loan_limit', 'approv_in_adv', 'submission_of_application', 'age', 'loan_purpose', 'Neg_ammortization']
dm = RAW[drop_cols].isna().any(axis=1)
print(f'dropped {int(dm.sum())} rows, default rate {RAW.Status[dm].mean():.3f} vs {RAW.Status.mean():.3f} overall')


=== 6. Rows removed by dropna filters ===
dropped 4728 rows, default rate 0.293 vs 0.246 overall
